[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Declarative Models


## What you will be able to do

Write a database's tables as Python classes with `DeclarativeBase`, `Mapped` and `mapped_column`,
read the `CREATE TABLE` each class becomes, and say how an annotation decides a column's type and
whether it may be empty. Query with the classes, and tell a result of objects from a result of rows.
Give a class behavior of its own, map a hierarchy of classes to one table or to several, and
recognize the annotation the ORM cannot map, the class with no primary key, and the subclass that
quietly shares its parent's table.


## The idea

### The problem

The **Tables and Metadata** notebook described the college's tables as `Table` objects, and every
query since has handed back rows: tuples with names. The registrar's application works with
students, courses and enrollments, which are things with behavior. A course's level comes from its
code, a grade is worth a number of points, and a transcript belongs to a student. With rows, that
behavior lives in functions scattered around the program, each of which takes a tuple and has to know
which position holds what.

The obvious fix is a class for every table, with a method for every behavior, and code that copies
each row into an object and each object back into a row. Now every table is described twice, once
as a `Table` and once as a class, and the copying code is a third place where a column added to one
of them has to be added to the others. The ORM writes the class once, and builds the table from it.

### What a mapped class is

> A **mapped class** is a Python class whose objects are the rows of a table. It inherits from a
> base class made by subclassing **`DeclarativeBase`**, names its table in **`__tablename__`**, and
> declares every column as an attribute annotated with **`Mapped[...]`**: `Mapped[int]`,
> `Mapped[str]`, `Mapped[date]`. The Python type in the annotation chooses the column's type, and
> `Mapped[str | None]` allows `NULL` where `Mapped[str]` does not. **`mapped_column()`** adds what
> the annotation cannot say: a primary key, a length, a foreign key, a default. The base's
> **`metadata`** collects the `Table` that every class builds, which is also the class's
> **`__table__`**.

### Why it works that way

- **The class is the table.** Declaring the class builds a `Table` in `Base.metadata`, so
  `create_all`, the naming convention and reflection from the **Tables and Metadata** notebook work
  unchanged.
- **The annotation is the schema.** `Mapped[date]` makes a `Date` column, `Mapped[str | None]` a
  column that may be `NULL`, and a type the ORM has no column type for is an error when the class is
  declared, not when a query runs.
- **The class attribute is a column, and the object's attribute is a value.** A condition such as
  `Student.program == "History"` is built from the class, exactly as
  `students.c.program == "History"` was built from the table, and `student.program` is one student's
  program.
- **Selecting a class returns objects.** `select(Student)` returns `Student` objects, which
  `session.scalars()` hands over one by one, while `select(Student.name)` returns rows, as before.
- **A mapped class is still a Python class.** It can have methods, properties and a `__repr__`, and
  the ORM gives it an `__init__` that takes every mapped attribute by keyword.
- **A class hierarchy can share a table or split it.** In single table inheritance, every subclass
  lives in the parent's table, told apart by a column that names the kind of row. In joined table
  inheritance, every subclass adds a table of its own, joined to the parent's by its primary key.

### Where this shows up

SQLModel, the subject of the **SQLModel, Deep Dive** guide, is a declarative base whose classes are
also Pydantic models. Flask-SQLAlchemy's `db.Model` is a declarative base with the session wired to
the web request. Alembic's autogenerate compares `Base.metadata` with a database to write a
migration, and **Column Types**, the notebook after this one, is about the annotations: which
Python type becomes which column, and what SQLite does with each. The **Object-Oriented Python**
guide covers the classes, properties and inheritance that a mapped class is made of.

### What this notebook covers

- A base, a first mapped class, and the table it builds
- The college's five tables as classes
- Objects or rows: what `select` returns for a class and for its columns
- How the annotation decides whether a column may be `NULL`
- Methods and properties on a mapped class
- Inheritance: one table for a hierarchy, or a table for every class
- Which to write: a `Table` or a class, and which kind of inheritance
- The dean's list, finished
- Five errors, from the annotation the ORM cannot map to the row mistaken for an object

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class Course(Base):
    __tablename__ = "courses"

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10))
    credits: Mapped[int]


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([Course(code="BIO-101", credits=4), Course(code="STA-200", credits=3)])
    for course in session.scalars(select(Course).where(Course.credits > 3)):
        print(type(course).__name__, course.code, course.credits)
```

```
Course BIO-101 4
```

A class with three annotated attributes became a table, the table was created from `Base.metadata`,
and the query returned a `Course` object, not a row. `Course.credits > 3` built the condition from
the class itself.


## Setup

Eight imports, the college built from its `MetaData`, and a helper that prints a `CREATE TABLE`.

- `sqlalchemy` is the library itself, and the cell prints its version
- `DeclarativeBase`, `Mapped`, `mapped_column` and `Session`, from `sqlalchemy.orm`, map classes to
  tables and read objects back
- `select`, `func`, `insert`, `inspect` and `text`, `create_engine` and `event`, and what describes a
  table, `MetaData`, `Table`, `Column`, the types, `ForeignKey` and the constraints, from
  `sqlalchemy`; `JSON` is a column type that Common errors uses
- `CreateTable`, from `sqlalchemy.schema`, turns a table into its `CREATE TABLE` statement
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what a `Date` column takes and returns
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds `scratch/college.db` as the **SQL Expressions** notebook did, from `college`, the five
tables of the **Tables and Metadata** notebook, and `show_ddl` is that notebook's helper. This
notebook writes the same five tables as classes, and the classes read the database Setup built.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (JSON, CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table,
                        UniqueConstraint, create_engine, event, func, insert, inspect, select, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}

def show_ddl(table, engine):
    """Print the CREATE TABLE statement a Table becomes on an engine's database."""
    for line in str(CreateTable(table).compile(engine)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### A base, a first mapped class, and the table it builds

A base class comes first, a subclass of `DeclarativeBase` with nothing in it, and every mapped class
inherits from it. The class names its table and annotates its columns:


In [2]:
class SketchBase(DeclarativeBase):
    pass


class Student(SketchBase):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]


print(type(Student.__table__).__name__, "|", Student.__table__.c.keys())
print("the same Table in the base's metadata:", SketchBase.metadata.tables["students"] is Student.__table__)
show_ddl(Student.__table__, engine)

ana = Student(name="Ana Reyes", email="areyes@college.edu", program="Biology", started_on=date(2024, 8, 26))
print(ana.name, "|", ana.started_on, "| id:", ana.id)


Table | ['id', 'name', 'email', 'program', 'started_on']
the same Table in the base's metadata: True
    CREATE TABLE students (
        id INTEGER NOT NULL,
        name VARCHAR(100) NOT NULL,
        email VARCHAR(200) NOT NULL,
        program VARCHAR(50) NOT NULL,
        started_on DATE NOT NULL,
        PRIMARY KEY (id),
        UNIQUE (email)
    )
Ana Reyes | 2024-08-26 | id: None


Declaring the class built a `Table` with one column for every annotated attribute, in the order they
were written, and put it in `SketchBase.metadata`. `Mapped[int]` became `INTEGER`, `Mapped[date]`
became `DATE`, and every column is `NOT NULL`, because none of the annotations allowed `None`.
`mapped_column` supplied the primary key, the lengths and the unique email. The ORM also wrote an
`__init__` that takes every column by keyword, and `ana`, made with it, is an object in memory with
no row in any database yet, so her `id` is `None`. Saving her is the business of **The Session**
notebook.

### The college's five tables as classes

The whole college, in one cell with one base, so that running the cell again starts from a fresh
base. The base carries the naming convention of the **Tables and Metadata** notebook in its own
`metadata`, `__table_args__` holds the constraints that belong to the table rather than a column, and
`ForeignKey` goes inside `mapped_column`. Every class gets a `__repr__`, since a mapped object prints
as its memory address without one, and two of them get a property:


In [3]:
GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    @property
    def level(self):
        """100 for an introductory course, 200 for the next, read from the number in the code."""
        return int(self.code.split("-")[1]) // 100 * 100

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


report = inspect(engine)
for cls in (Student, Course, Term, Section, Enrollment):
    in_database = [column["name"] for column in report.get_columns(cls.__tablename__)]
    print(f"{cls.__name__:<11} {cls.__table__.c.keys()} matches the database: {cls.__table__.c.keys() == in_database}")


Student     ['id', 'name', 'email', 'program', 'started_on'] matches the database: True
Course      ['id', 'code', 'title', 'department', 'credits'] matches the database: True
Term        ['id', 'name', 'starts_on'] matches the database: True
Section     ['id', 'course_id', 'term_id', 'capacity'] matches the database: True
Enrollment  ['student_id', 'section_id', 'status', 'grade'] matches the database: True


Five classes, and the table each one builds has exactly the columns of the table Setup made. The
classes do not create anything: the tables are already there, and `Base.metadata.create_all(engine)`
would find all five and create nothing, as the **Tables and Metadata** notebook showed. `Enrollment`
has a primary key of two columns, since both carry `primary_key=True`. `GRADE_POINTS` is the table of
grade points that `Enrollment.grade_points` reads.

### Objects or rows

A `Session` runs statements for the ORM, the way a connection runs them for Core, and
`session.scalars()` hands over the first thing in every row. For `select(Student)`, that is a
`Student`:


In [4]:
with Session(engine) as session:
    history = session.scalars(select(Student).where(Student.program == "History").order_by(Student.name)).all()
    print(history)
    first = history[0]
    print(type(first).__name__, "|", first.name, "|", first.started_on, "| id", first.id)

    pairs = session.execute(select(Student.name, Student.program).where(Student.id <= 2)).all()
    print(type(pairs[0]).__name__, pairs)

    row = session.execute(select(Student).where(Student.id == 3)).one()
    print(type(row).__name__, row, "| the object inside it:", row.Student)


[Student("Aoife O'Brien", 'History'), Student('Elena Petrova', 'History'), Student('Jonas Berg', 'History'), Student('Olivia Brandt', 'History'), Student('Tara Nilsen', 'History')]
Student | Aoife O'Brien | 2024-08-26 | id 25
Row [('Ana Reyes', 'Biology'), ('Ben Okafor', 'Computer Science')]
Row (Student('Chloe Martin', 'Mathematics'),) | the object inside it: Student('Chloe Martin', 'Mathematics')


`select(Student)` returned five `Student` objects, printed by their `__repr__`, with every column an
attribute and `started_on` a `date`. `select(Student.name, Student.program)` asked for two columns,
not a class, and got rows, exactly as in the **Reading Results** notebook. The last query shows what
`scalars()` saves: `session.execute(select(Student))` returns rows too, each holding one `Student`,
reached by position or by the class's name, `row.Student`. **The Session** notebook covers what a
session does with the objects it hands out.

### How the annotation decides whether a column may be NULL

`Enrollment.grade` is `Mapped[str | None]`, and `Enrollment.status` is `Mapped[str]` with a default
in the table. The `CREATE TABLE` shows what each annotation became:


In [5]:
show_ddl(Enrollment.__table__, engine)

with Session(engine) as session:
    spring = session.scalars(select(Enrollment).where(Enrollment.student_id == 3, Enrollment.section_id > 30)).all()
print(spring)


    CREATE TABLE enrollments (
        student_id INTEGER NOT NULL,
        section_id INTEGER NOT NULL,
        status VARCHAR(20) DEFAULT 'enrolled' NOT NULL,
        grade VARCHAR(2),
        CONSTRAINT pk_enrollments PRIMARY KEY (student_id, section_id),
        CONSTRAINT ck_enrollments_status_known CHECK (status IN ('enrolled', 'completed', 'withdrawn')),
        CONSTRAINT fk_enrollments_student_id_students FOREIGN KEY(student_id) REFERENCES students (id),
        CONSTRAINT fk_enrollments_section_id_sections FOREIGN KEY(section_id) REFERENCES sections (id)
    )
[Enrollment(student 3, section 33, None), Enrollment(student 3, section 37, None), Enrollment(student 3, section 40, None)]


`grade VARCHAR(2)` has no `NOT NULL`, because the annotation allows `None`, and a Spring 2026
enrollment has no grade yet, so `grade` is `None` in every one of Chloe Martin's. Every `Mapped[...]`
without `| None` became `NOT NULL`. The annotation is the ordinary way to say it, and
`mapped_column(nullable=...)` overrides it when the two must differ, such as a column the database
fills in itself.

### Methods and properties on a mapped class

A mapped class is a Python class, so behavior that belongs to a course or an enrollment can live on
it. `Course.level` reads the level from the code, and `Enrollment.grade_points` turns a grade into
its points:


In [6]:
with Session(engine) as session:
    for course in session.scalars(select(Course).where(Course.department == "Computer Science").order_by(Course.code)):
        print(course, "| level", course.level)
    finished = select(Enrollment).where(Enrollment.student_id == 6, Enrollment.grade.is_not(None))
    for enrollment in session.scalars(finished.order_by(Enrollment.section_id)):
        print(enrollment, "| worth", enrollment.grade_points)


Course('CSC-101', 3) | level 100
Course('CSC-201', 3) | level 200
Enrollment(student 6, section 22, 'B') | worth 3.0
Enrollment(student 6, section 25, 'C') | worth 2.0
Enrollment(student 6, section 29, 'A') | worth 4.0


The properties are plain Python, computed from the attributes every time they are read, and the
database knows nothing about them: a query cannot filter on `Course.level`, because there is no such
column. Felix Wagner's three finished courses are worth 3.0, 2.0 and 4.0 points a credit.

### Inheritance: one table for a hierarchy, or a table for every class

The college's staff are people of several kinds. In single table inheritance, every kind lives in one
table, `people`, and a column, here `kind`, says which class a row belongs to. `__mapper_args__`
names that column in the parent, and every class's own value for it. A column only one kind has must
allow `NULL`, since the other kinds' rows leave it empty:


In [7]:
class StaffBase(DeclarativeBase):
    pass


class Person(StaffBase):
    __tablename__ = "people"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    kind: Mapped[str] = mapped_column(String(20))

    __mapper_args__ = {"polymorphic_on": "kind", "polymorphic_identity": "person"}

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Instructor(Person):
    department: Mapped[str | None] = mapped_column(String(50))
    __mapper_args__ = {"polymorphic_identity": "instructor"}


class Advisor(Person):
    office: Mapped[str | None] = mapped_column(String(20))
    __mapper_args__ = {"polymorphic_identity": "advisor"}


staff = college_engine()                          # a database in memory, for the staff alone
StaffBase.metadata.create_all(staff)
with Session(staff) as session:
    session.add_all([Person(name="Ada Registrar"), Instructor(name="Dr. Okafor", department="Biology"),
                     Advisor(name="Ms. Lin", office="B-204")])
    session.commit()
    print("everyone:      ", session.scalars(select(Person).order_by(Person.id)).all())
    print("instructors:   ", session.scalars(select(Instructor)).all())
    rows = session.execute(text("SELECT id, name, kind, department, office FROM people ORDER BY id")).all()
    print("the table:     ", rows)


everyone:       [Person('Ada Registrar'), Instructor('Dr. Okafor'), Advisor('Ms. Lin')]
instructors:    [Instructor('Dr. Okafor')]
the table:      [(1, 'Ada Registrar', 'person', None, None), (2, 'Dr. Okafor', 'instructor', 'Biology', None), (3, 'Ms. Lin', 'advisor', None, 'B-204')]


One table held all three, and `kind` recorded the class of every row, so `select(Person)` returned a
`Person`, an `Instructor` and an `Advisor`, and `select(Instructor)` only the instructor, since the
ORM adds a condition on `kind` to any query for a subclass. `session.add_all` and `session.commit()`
save new objects, the subject of **The Session** notebook. In joined table inheritance, a subclass
gets a table of its own, whose primary key is also a foreign key to the parent's:


In [8]:
class Visitor(Person):
    __tablename__ = "visitors"

    id: Mapped[int] = mapped_column(ForeignKey("people.id"), primary_key=True)
    host: Mapped[str] = mapped_column(String(100))

    __mapper_args__ = {"polymorphic_identity": "visitor"}


StaffBase.metadata.create_all(staff)
with Session(staff) as session:
    session.add(Visitor(name="Prof. Haddad", host="Dr. Okafor"))
    session.commit()
    print(" ".join(str(select(Visitor).compile(staff)).split()))
    print(session.scalars(select(Visitor)).all(), "| hosted by", session.scalars(select(Visitor)).one().host)
staff.dispose()


SELECT visitors.id, people.id AS id_1, people.name, people.kind, visitors.host FROM people JOIN visitors ON people.id = visitors.id
[Visitor('Prof. Haddad')] | hosted by Dr. Okafor


`create_all` added `visitors` and left `people` alone, and a visitor's row is split between the two:
the name and the kind in `people`, the host in `visitors`, and `select(Visitor)` joins them on the
id. A hierarchy can mix the two kinds, as this one now does.

### Which to write: a Table or a class, and which kind of inheritance

| Use | When | Why |
|---|---|---|
| a mapped class | rows that are things with behavior, in an application | objects with attributes and methods, and one description of the table |
| a `Table` | reports, loads and other work on rows, or a table no class needs | no objects to build, and nothing but Core in the way |
| `Mapped[X \| None]` | a column that may be empty | the annotation and the `NULL` agree |
| single table inheritance | kinds that differ in a column or two | one table, and no join to read any of them |
| joined table inheritance | kinds with many columns of their own | no column that most rows leave empty, at the price of a join |

The default for an application is a mapped class for every table, with `Mapped[X | None]` for every
column that may be empty. Single table inheritance is the default for a hierarchy until the
subclasses' own columns outnumber the shared ones.

### The dean's list, finished

The pieces of this notebook in one function. `deans_list` selects three classes at once, joins them
along the foreign keys, uses `Enrollment.grade_points` and `Course.credits` to work out every
student's average for a term, and keeps the students at or above the bar, as objects:


In [9]:
def deans_list(session, term, minimum=3.0):
    """The students whose grade point average for a term is at least `minimum`, best first, then by name."""
    graded = (
        select(Student, Enrollment, Course)
        .join(Enrollment)
        .join(Section)
        .join(Course)
        .join(Term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
    )
    points, credits = {}, {}
    for student, enrollment, course in session.execute(graded):
        points[student] = points.get(student, 0) + enrollment.grade_points * course.credits
        credits[student] = credits.get(student, 0) + course.credits
    averages = {student: round(points[student] / credits[student], 2) for student in points}
    chosen = [(average, student) for student, average in averages.items() if average >= minimum]
    return sorted(chosen, key=lambda pair: (-pair[0], pair[1].name))



with Session(engine) as session:
    for term in ("Spring 2025", "Fall 2025"):
        print(term)
        for average, student in deans_list(session, term):
            print(f"    {average:.2f}  {student}")


Spring 2025
    3.10  Student('Ana Reyes', 'Biology')
    3.10  Student('Keiko Tanaka', 'Biology')
Fall 2025
    3.01  Student('Grace Lin', 'Computer Science')
    3.01  Student('Quinn Harper', 'Computer Science')
    3.00  Student('Felix Wagner', 'Biology')
    3.00  Student('Pavel Novak', 'Biology')


`select(Student, Enrollment, Course)` returned rows of three objects, which the loop unpacked, and
the joins found their `ON` clauses in the foreign keys, as they did between tables in the
**SQL Expressions** notebook. The students are dictionary keys, which works because the session
hands back one object for every row, however many times the row appears, a promise that
**The Identity Map** notebook takes apart. Nobody reached 3.5 in either term, so the bar is 3.0.

### Where each part came from

| In `deans_list` | What it relies on | The section that showed it |
|---|---|---|
| `select(Student, Enrollment, Course)` | a query that returns objects, several to a row | Objects or rows |
| `.join(Enrollment).join(Section)` | joins along the foreign keys that `mapped_column` declared | The college's five tables as classes |
| `Enrollment.grade.is_not(None)` | a column the annotation allowed to be empty | How the annotation decides whether a column may be NULL |
| `enrollment.grade_points` | behavior on the mapped class | Methods and properties on a mapped class |
| printing `student` | the class's `__repr__` | The college's five tables as classes |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/07-declarative-models-solutions.ipynb).

**1.** In a base of your own, map a `Room` class to a `rooms` table: an id, a building and a number,
both required, and a number of seats that may be unknown. Print its `CREATE TABLE`.


In [10]:
# your code here


**2.** In a session, list the courses worth 4 credits as `Course` objects, and then as rows of code
and title. Print the type of the first item of each.


In [11]:
# your code here


**3.** Print Chloe Martin's Fall 2025 enrollments, student 3 and sections 21 to 30, with the grade
points of each.


In [12]:
# your code here


**4.** Count the students in every program with `select(Student.program, func.count())`, and say why
the result holds rows rather than objects.


In [13]:
# your code here


**5.** Add a `Librarian` kind to the staff, with no columns of its own, in a new database in memory.
Save one of each kind, and show that `select(Person)` returns every kind as its own class.


In [14]:
# your code here


**6.** Print the dean's list for Fall 2024 with a bar of 2.8.


In [15]:
# your code here


## Common errors

### sqlalchemy.orm.exc.MappedAnnotationError: Could not locate SQLAlchemy Core type when resolving for Python type indicated by 'list[str]' inside the Mapped[] annotation for the 'phones' attribute; the type object is not resolvable by the registry


In [16]:
class ContactBase(DeclarativeBase):
    pass


class Contact(ContactBase):
    __tablename__ = "contacts"

    id: Mapped[int] = mapped_column(primary_key=True)
    phones: Mapped[list[str]]


MappedAnnotationError: Could not locate SQLAlchemy Core type when resolving for Python type indicated by 'list[str]' inside the Mapped[] annotation for the 'phones' attribute; the type object is not resolvable by the registry

The ORM looks the annotation's Python type up in a table of its own, the registry, to choose a
column type, and a list of strings has no column type: a relational column holds one value. There
are two answers. A list that belongs to a row is usually a table of its own, one row for every
phone, which is what relationships are for, in the **Relationships** notebook. Or the list can be
stored whole, as JSON, and then `mapped_column` names the column type the annotation could not:


In [17]:
class ContactBase(DeclarativeBase):
    pass


class Contact(ContactBase):
    __tablename__ = "contacts"

    id: Mapped[int] = mapped_column(primary_key=True)
    phones: Mapped[list[str]] = mapped_column(JSON)


contacts = college_engine()
ContactBase.metadata.create_all(contacts)
with Session(contacts) as session:
    session.add(Contact(phones=["555-0100", "555-0199"]))
    session.commit()
    print(session.scalars(select(Contact.phones)).one())
contacts.dispose()


['555-0100', '555-0199']


### sqlalchemy.exc.ArgumentError: Mapper Mapper[Waitlist(waitlist)] could not assemble any primary key columns for mapped table 'waitlist'


In [18]:
class WaitBase(DeclarativeBase):
    pass


class Waitlist(WaitBase):
    __tablename__ = "waitlist"

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"))
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"))


ArgumentError: Mapper Mapper[Waitlist(waitlist)] could not assemble any primary key columns for mapped table 'waitlist'

The ORM tells objects apart by their primary key, so a mapped class must have one, and this one does
not. A student is on a section's waitlist once at most, so the pair of columns is the key: mark both
with `primary_key=True`:


In [19]:
class WaitBase(DeclarativeBase):
    pass


class Waitlist(WaitBase):
    __tablename__ = "waitlist"

    student_id: Mapped[int] = mapped_column(primary_key=True)
    section_id: Mapped[int] = mapped_column(primary_key=True)


print(Waitlist.__table__.primary_key.columns.keys())


['student_id', 'section_id']


### sqlalchemy.orm.exc.MappedAnnotationError: Type annotation for "Room.building" can't be correctly interpreted for Annotated Declarative Table form.


In [20]:
class RoomBase(DeclarativeBase):
    pass


class Room(RoomBase):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    building: str


MappedAnnotationError: Type annotation for "Room.building" can't be correctly interpreted for Annotated Declarative Table form.  ORM annotations should normally make use of the ``Mapped[]`` generic type, or other ORM-compatible generic type, as a container for the actual type, which indicates the intent that the attribute is mapped. Class variables that are not intended to be mapped by the ORM should use ClassVar[].  To allow Annotated Declarative to disregard legacy annotations which don't use Mapped[] to pass, set "__allow_unmapped__ = True" on the class or a superclass this class. (Background on this error at: https://sqlalche.me/e/20/zlpr)

`building: str` is an ordinary annotation, and the ORM maps only attributes inside `Mapped[...]`. It
refuses the plain one rather than guess, since the class might have meant it as a column or as an
ordinary attribute, and the rest of the message, below the traceback, says how to mark either. A
column is `Mapped[str]`:


In [21]:
class RoomBase(DeclarativeBase):
    pass


class Room(RoomBase):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    building: Mapped[str]


print(Room.__table__.c.keys())


['id', 'building']


### No error, and everyone an instructor: a subclass with no table and no identity


In [22]:
class TeamBase(DeclarativeBase):
    pass


class Member(TeamBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Lecturer(Member):                          # no __tablename__, and no polymorphic_identity
    department: Mapped[str | None]


team = college_engine()
TeamBase.metadata.create_all(team)
with Session(team) as session:
    session.add_all([Member(name="Ada Registrar"), Lecturer(name="Dr. Okafor", department="Biology")])
    session.commit()
    print("the table:   ", Lecturer.__table__.name, Member.__table__.c.keys())
    print("members:     ", session.scalars(select(Member).order_by(Member.id)).all())
    print("lecturers:   ", session.scalars(select(Lecturer).order_by(Lecturer.id)).all())
team.dispose()


the table:    members ['id', 'name', 'department']
members:      [Member('Ada Registrar'), Member('Dr. Okafor')]
lecturers:    [Lecturer('Ada Registrar'), Lecturer('Dr. Okafor')]


With no `__tablename__`, `Lecturer` joined its parent's table, and its `department` became a column
of `members`. That is single table inheritance, but with nothing to tell the kinds apart: no column
records which class a row belongs to. So `select(Member)` brought Dr. Okafor back as a `Member`, and
`select(Lecturer)` brought back both people as lecturers, Ada Registrar included. Nothing raised,
because every step is legal. Name a discriminator column in the parent and an identity in every
class, as the staff hierarchy above does, or give the subclass a table of its own.

### AttributeError: name


In [23]:
with Session(engine) as session:
    chloe = session.execute(select(Student).where(Student.id == 3)).one()
    print(chloe.name)


AttributeError: name

`session.execute(select(Student))` returns rows, and every row holds a `Student`, so `chloe` is a row
with one element, and a row has no column called `name`. The message is only the name of the
attribute, which makes this one easy to misread. `session.scalars()` takes the object out of each
row:


In [24]:
with Session(engine) as session:
    chloe = session.scalars(select(Student).where(Student.id == 3)).one()
    print(chloe.name, "|", chloe.program)


Chloe Martin | Mathematics


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [25]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A class that inherits from a `DeclarativeBase` subclass, with a `__tablename__` and `Mapped[...]`
  attributes, is a table: declaring it builds a `Table` in `Base.metadata`.
- The annotation chooses the column type, and `Mapped[X | None]` is the column that may be `NULL`;
  `mapped_column` adds keys, lengths and defaults.
- `select(Student)` returns objects, which `session.scalars()` hands over, and `select(Student.name)`
  returns rows.
- A mapped class keeps its methods, properties and `__repr__`, and every mapped class needs a primary
  key.
- Single table inheritance tells kinds apart by a column named in `__mapper_args__`, and joined table
  inheritance gives every subclass a table of its own.


## What is next

The **Column Types** notebook looks inside the annotations: which Python type becomes which column,
naive and aware datetimes, the precision SQLite quietly loses, and the type that only accepts a
datetime.


---

&#8592; **Previous:** [SQL Expressions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/06-sql-expressions.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/08-column-types.ipynb) &#8594;
